# **Diplomado IA: Inteligencia Artificial I - Parte 1**. <br> Práctico 13.2: Seq2Seq y Mecanismos de Atención
---
---

**Profesores:**
- Gabriel Sepúlveda

**Ayudante:**
-
---
---

# **Instrucciones Generales**

El siguiente práctico se debe realizar de forma individual. El formato de entrega es el **archivo .ipynb con todas las celdas ejecutadas**, en el cual deberá completar las actividades propuestas.

**Fecha de entrega:** domingo 7 de diciembre de 2025, 23:59 hrs.

**Nombre alumno:**

In [ ]:
!pip install torchtext==0.6.0

# Import Base Dependencies

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
try:
    from tqdm.notebook import tqdm, trange
except:
    from tqdm import tqdm, trange

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data.dataset import Dataset
from torchtext import data, datasets, vocab

In [ ]:
# for reproducibility
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using {DEVICE}')

# General Utilities

In [ ]:
def num_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Sequence to sequence (seq2seq) with attention


![](https://distill.pub/2016/augmented-rnns/assets/rnn_attentional_02.svg)

from [distill.pub](https://distill.pub/2016/augmented-rnns/)

## Encoder

(basically unchanged from non-attentional variant)

In [ ]:
class EncoderModule(nn.Module):
    def __init__(self, embedding_size, hidden_size, source_vocab_size):
        super().__init__()
        self.embeddings_table = nn.Embedding(source_vocab_size,
                                             embedding_size,
                                             padding_idx=0)
        self.embedding_size = embedding_size
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size=embedding_size,
                            hidden_size=hidden_size,
                            bidirectional=True,
                            batch_first=True)

    # forward pass
    def forward(self, src_sentences):
        """
        src_sentences shape (batch, seq_len)
        """
        # embed input sequences (batch, seq_len, embed_size)
        embedded = self.embeddings_table(src_sentences)

        # run lstm on full input sequence Tuple(Tensor, Tensor)
        all_enc_hidden_states, hidden_state = self.lstm(embedded)

        return hidden_state, all_enc_hidden_states

## Decoder

Same as decoder from last time, except we will concatenate the result of the attention operation to the hidden state to calculate output probs.

### Attention:


- We compute an **Alignment score** for previous decoder's hidden state (Query) $s_{t-1}$ and each encoder's hidden state (Key) $h_{j}$.
    <br>
    $$
        \hat{\alpha}_{t,j} = \text{score}(s_{t-1}, h_{j}) = V_{c}^{T} \cdot tanh \left( W_{c} \cdot s_{t-1} + U_{c} \cdot h_{j} \right)
    $$
    <br>

- We normalize the scores to obtain the **Attentional probabilities / weights**:
    <br>
    $$
        \alpha_{t,j} = Softmax(\hat{\alpha}_{t,j}) = \frac{\hat{\alpha}_{t,j}}{\sum_{k} \hat{\alpha}_{t,k}}
    $$
    <br>

- Finally, we can multiply each value by its weight to obtain the **Context vector**:
    <br>
    $$
        C = \sum_{i = 1}^{T} \alpha_{t,i} \cdot \lt \overrightarrow{h_{i}},\overleftarrow{h_{i}} \gt
    $$
    <br>



### Attention Module

In [ ]:
class AttentionModule( nn.Module ):

  def __init__( self, units ):
    super( AttentionModule, self ).__init__()
    self.W = nn.Linear( units, units )
    self.U = nn.Linear( units, units )
    self.V = nn.Linear( units, 1 )

  def forward( self, query, values ):
    # query hidden state shape == (batch_size, hidden size)
    # query_with_time_axis shape == (batch_size, 1, hidden size)
    # values shape == (batch_size, max_len, hidden size)
    # we are doing this to broadcast addition along the time axis to calculate the score
    query_with_time_axis = query.unsqueeze( 1 )

    # score shape == (batch_size, max_length, 1)
    # we get 1 at the last axis because we are applying score to self.V
    # the shape of the tensor before applying self.V is (batch_size, max_length, units)
    score = self.V( torch.tanh( self.W(query_with_time_axis) + self.U(values) ) )

    # attention_weights shape == (batch_size, max_length, 1)
    attention_weights = torch.softmax( score, dim = 1 )

    # context_vector shape after sum == (batch_size, hidden_size)
    context_vector = attention_weights * values
    context_vector = torch.sum( context_vector, 1 )

    return context_vector

## Decoder Module

In [ ]:
class DecoderModule(nn.Module):
    def __init__(self, embedding_size, hidden_size, start_idx, dst_vocab_size):
        super().__init__()
        self.embeddings_table = nn.Embedding(dst_vocab_size,
                                             embedding_size,
                                             padding_idx=0)
        self.start_idx = torch.tensor(start_idx).to(DEVICE)
        self.hidden_size = hidden_size

        self.h2o = torch.nn.Linear(2*hidden_size, dst_vocab_size)
        self.lstm_cell = torch.nn.LSTMCell(input_size=embedding_size,
                                           hidden_size=hidden_size)

        self.attention = AttentionModule(hidden_size)

    # forward pass
    def forward(self, final_encoder_hidden_state, all_enc_hidden_states, max_output_length):
        """
        final_encoder_hidden_state: Tuple(
            Tensor: (batch, hidden_size)
            Tensor: (batch, hidden_size)
        )
        all_enc_hidden_states: Tensor (batch, max_len_enc, hidden_size)
        max_output_length: int

        returns Tensor: (batch, max_output_length, dst_vocab_size)
        """
        batch_size = final_encoder_hidden_state[0].shape[0]

        out = []
        # initial decoder hidden state is final encoder hidden state
        state = final_encoder_hidden_state

        # decoding loop (decode one output word at a time)
        y_t = self.embeddings_table(self.start_idx.repeat(batch_size))
        for i in range(max_output_length):
            (hidden_state, cell_state)  = self.lstm_cell(y_t, state)

            # compute context vector using attention
            context_vector = self.attention(
                query=hidden_state,
                #keys=all_enc_hidden_states,
                values=all_enc_hidden_states,
            )

            # calculate output probs
            concat_input = torch.cat((hidden_state, context_vector), -1)
            P_t = self.h2o(concat_input)    # (batch, dst_vocab_size)
            out.append(P_t)

            _, max_indices = P_t.max(dim=1)
            y_t = self.embeddings_table(max_indices)

            state = (hidden_state, cell_state)

        return torch.stack(out, dim=1)

## Data

SCAN dataset ([link](https://github.com/brendenlake/SCAN))

Translates english commands to robot actions:

| Input |   | Output |
| --- | --- | --- |
| jump	|| JUMP |
| jump left	|| LTURN JUMP|
| jump around right	|| RTURN JUMP RTURN JUMP RTURN JUMP RTURN JUMP |


### Download dataset

In [ ]:
import requests

BASE_PATH = 'https://gist.githubusercontent.com/ceyzaguirre4/707273ad9fc1729ebe2daac442a8f5a8/raw/'

def download_scan(split):
    filename = f'{split}.txt'
    url = BASE_PATH + filename
    response = requests.get(url)

    with open(filename, 'w') as out_file:
        out_file.write(response.text)

download_scan('tasks_test_simple')
download_scan('tasks_train_simple')

In [ ]:
!head tasks_test_simple.txt

### Load data

In [ ]:
def tokenize(text):
    return text.strip().lower().split()

In [ ]:
SOURCE = data.Field(tokenize=tokenize, unk_token=None, batch_first=True)
TARGET = data.Field(tokenize=tokenize, unk_token=None, batch_first=True)

train, test = data.TabularDataset.splits(".",
                                         train="tasks_train_simple.txt",
                                         test="tasks_test_simple.txt",
                                         format="tsv",
                                         fields=[
                                                 ("source", SOURCE),
                                                 ("target", TARGET)],
                                         skip_header=False)

SOURCE.build_vocab(train)
TARGET.build_vocab(train)

train_iter, test_iter = data.BucketIterator.splits((train, test),
                                                   batch_sizes=(128, 128),
                                                   device=DEVICE,
                                                   sort_key=lambda x: len(x.target))

## Train Utils

In [ ]:
def train_one_epoch(model, dataloader, optimizer, history):
    model.train()
    for batch in dataloader:
        # forward pass
        y_gt = batch.target
        batch_size, max_ouput_len = batch.target.shape
        y_pred = model(batch.source, max_ouput_len)

        loss = F.cross_entropy(y_pred.view(batch_size*max_ouput_len, -1),
                               y_gt.view(-1))

        history['train_loss'].append(loss.item())

        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
def eval_one_epoch(model, dataloader, optimizer, history):
    model.eval()
    history['eval_acc'].append(0)
    history['eval_loss'].append(0)
    with torch.no_grad():
        for batch in dataloader:
            # forward pass
            y_gt = batch.target
            batch_size, max_ouput_len = batch.target.shape
            y_pred = model(batch.source, max_ouput_len)

            loss = F.cross_entropy(y_pred.view(batch_size*max_ouput_len, -1),
                                y_gt.view(-1))
            accuracy = (y_pred.argmax(dim=2) == y_gt).float().mean()

            history['eval_acc'][-1] += (accuracy / len(dataloader)).item()
            history['eval_loss'][-1] += (loss / len(dataloader)).item()

In [ ]:
def train_model(model, train_dataloader, test_dataloader, optimizer, epochs):
    history = defaultdict(list)
    for epoch in trange(epochs):
        train_one_epoch(model, train_dataloader, optimizer, history)
        eval_one_epoch(model, test_dataloader, optimizer, history)
    return history

## Full Model

In [ ]:
class SeqToSeq(nn.Module):
    def __init__(self, embedding_size, hidden_size, source_vocab_size, dest_vocab_size, start_idx):
        super().__init__()
        self.hidden_size = hidden_size

        self.encoder = EncoderModule(embedding_size, hidden_size, source_vocab_size)
        self.decoder = DecoderModule(embedding_size, 2*hidden_size, start_idx, dest_vocab_size)

    def reshape_enc_states(self, enc_hidden_states):
        """
        enc_hidden_states: Tuple(
            hidden_state: Tensor (2, batch, hidden_size)
            cell_state: Tensor (2, batch, hidden_size)
        )

        returns Tuple(
            hidden_state: Tensor (batch, 2*hidden_size)
            cell_state: Tensor (batch, 2*hidden_size)
        )
        """
        def _reshape_enc_state(enc_hidden_state):
            enc_hidden_state = enc_hidden_state\
                .permute(1, 0, 2)\
                .reshape(-1, 2*self.hidden_size)
            return enc_hidden_state

        hidden_state, cell_state = enc_hidden_states

        hidden_state = _reshape_enc_state(hidden_state)
        cell_state = _reshape_enc_state(cell_state)

        return (hidden_state, cell_state)

    def forward(self, src_sentences, max_output_length):
        # run encoder
        encoder_hidden_states, all_enc_hidden_states = self.encoder(src_sentences)

        # reshape encoder state tensors for decoder
        encoder_hidden_states = self.reshape_enc_states(encoder_hidden_states)

        # run decoder (batch, max_output_length, dst_vocab_size)
        outputs = self.decoder(encoder_hidden_states, all_enc_hidden_states, max_output_length)

        return outputs

In [ ]:
# embeddings_size, hidden_size, source_vocab_size, dest_vocab_size, start_idx):
model = SeqToSeq(100, 150,
                 len(SOURCE.vocab),
                 len(TARGET.vocab)+1,
                 len(TARGET.vocab))
model.to(DEVICE)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
history = train_model(model, train_iter, test_iter, optimizer, 300)

In [ ]:
plt.plot(history['eval_acc'])

## Visualize attentions

### Hidden code for visualization

This code is repeated with the code above, but incorporates minor changes for visualization purposes.

In [ ]:
class AttentionModule( nn.Module ):

  def __init__( self, units ):
    super( AttentionModule, self ).__init__()
    self.W = nn.Linear( units, units )
    self.U = nn.Linear( units, units )
    self.V = nn.Linear( units, 1 )

  def forward( self, query, values ):
    # query hidden state shape == (batch_size, hidden size)
    # query_with_time_axis shape == (batch_size, 1, hidden size)
    # values shape == (batch_size, max_len, hidden size)
    # we are doing this to broadcast addition along the time axis to calculate the score
    query_with_time_axis = query.unsqueeze( 1 )

    # score shape == (batch_size, max_length, 1)
    # we get 1 at the last axis because we are applying score to self.V
    # the shape of the tensor before applying self.V is (batch_size, max_length, units)
    score = self.V( torch.tanh( self.W(query_with_time_axis) + self.U(values) ) )

    # attention_weights shape == (batch_size, max_length, 1)
    attention_weights = torch.softmax( score, dim = 1 )

    # context_vector shape after sum == (batch_size, hidden_size)
    context_vector = attention_weights * values
    context_vector = torch.sum( context_vector, 1 )

    return context_vector, attention_weights.squeeze()

In [ ]:
class DecoderModule(nn.Module):
    def __init__(self, embedding_size, hidden_size, start_idx, dst_vocab_size):
        super().__init__()
        self.embeddings_table = nn.Embedding(dst_vocab_size,
                                             embedding_size,
                                             padding_idx=0)
        self.start_idx = torch.tensor(start_idx).to(DEVICE)
        self.hidden_size = hidden_size

        self.h2o = torch.nn.Linear(2*hidden_size, dst_vocab_size)
        self.lstm_cell = torch.nn.LSTMCell(input_size=embedding_size,
                                           hidden_size=hidden_size)

        self.attention = AttentionModule(hidden_size)

    # forward pass
    def forward(self, final_encoder_hidden_state, all_enc_hidden_states, max_output_length):
        """
        final_encoder_hidden_state: Tuple(
            Tensor: (batch, hidden_size)
            Tensor: (batch, hidden_size)
        )
        all_enc_hidden_states: Tensor (batch, max_len_enc, hidden_size)
        max_output_length: int

        returns Tensor: (batch, max_output_length, dst_vocab_size)
        """
        batch_size = final_encoder_hidden_state[0].shape[0]

        out = []
        attentions = []
        # initial decoder hidden state is final encoder hidden state
        state = final_encoder_hidden_state

        # decoding loop (decode one output word at a time)
        y_t = self.embeddings_table(self.start_idx.repeat(batch_size))
        for i in range(max_output_length):
            (hidden_state, cell_state)  = self.lstm_cell(y_t, state)

            # compute context vector using attention
            context_vector, att = self.attention(
                query=hidden_state,
                #keys=all_enc_hidden_states,
                values=all_enc_hidden_states,
            )

            # calculate output probs
            concat_input = torch.cat((hidden_state, context_vector), -1)
            P_t = self.h2o(concat_input)    # (batch, dst_vocab_size)
            out.append(P_t)
            attentions.append(att)

            _, max_indices = P_t.max(dim=1)
            y_t = self.embeddings_table(max_indices)

            state = (hidden_state, cell_state)

        return torch.stack(out, dim=1), torch.stack(attentions, dim=1)

In [ ]:
class SeqToSeq(nn.Module):
    def __init__(self, embedding_size, hidden_size, source_vocab_size, dest_vocab_size, start_idx):
        super().__init__()
        self.hidden_size = hidden_size

        self.encoder = EncoderModule(embedding_size, hidden_size, source_vocab_size)
        self.decoder = DecoderModule(embedding_size, 2*hidden_size, start_idx, dest_vocab_size)

    def reshape_enc_states(self, enc_hidden_states):
        """
        enc_hidden_states: Tuple(
            hidden_state: Tensor (2, batch, hidden_size)
            cell_state: Tensor (2, batch, hidden_size)
        )

        returns Tuple(
            hidden_state: Tensor (batch, 2*hidden_size)
            cell_state: Tensor (batch, 2*hidden_size)
        )
        """
        def _reshape_enc_state(enc_hidden_state):
            enc_hidden_state = enc_hidden_state\
                .permute(1, 0, 2)\
                .reshape(-1, 2*self.hidden_size)
            return enc_hidden_state

        hidden_state, cell_state = enc_hidden_states

        hidden_state = _reshape_enc_state(hidden_state)
        cell_state = _reshape_enc_state(cell_state)

        return (hidden_state, cell_state)

    def forward(self, src_sentences, max_output_length):
        # run encoder
        encoder_hidden_states, all_enc_hidden_states = self.encoder(src_sentences)

        # reshape encoder state tensors for decoder
        encoder_hidden_states = self.reshape_enc_states(encoder_hidden_states)

        # run decoder (batch, max_output_length, dst_vocab_size)
        outputs, attentions = self.decoder(encoder_hidden_states, all_enc_hidden_states, max_output_length)

        return outputs, attentions

### Run trained model and save attentions

In [ ]:
# load pretrained
trained_state_dict = model.state_dict()
new_model = SeqToSeq(100, 150,
                len(SOURCE.vocab),
                len(TARGET.vocab)+1,
                len(TARGET.vocab))
new_model.to(DEVICE)
new_model.load_state_dict(trained_state_dict)

In [ ]:
new_model.eval()
with torch.no_grad():
    batch = next(iter(test_iter))
    # forward pass
    y_gt = batch.target
    batch_size, max_ouput_len = batch.target.shape
    y_pred, attentions = new_model(batch.source, max_ouput_len)
    predicted_words = y_pred.argmax(2).data.cpu().numpy()

## Visualize attentions

In [ ]:
import seaborn as sns

In [ ]:
def visualize(source, predicted, attentions):
    # map predictions to words
    pred_words = [TARGET.vocab.itos[predicted[i]] for i in range(len(predicted))]
    source_words = [SOURCE.vocab.itos[source[i]] for i in range(len(source))]

    # show attentions
    plt.figure(figsize=(14, 10))
    ax = sns.heatmap(attentions.cpu().numpy()[::], linewidth=1, vmin=0, vmax=1,
                        yticklabels=pred_words, xticklabels=source_words,
                        cmap="Reds")
    plt.yticks(rotation=0)
    plt.show()

In [ ]:
# index of element of batch that will be shown in visualization
batch_idx = 5

In [ ]:
visualize(batch.source[batch_idx],
          predicted_words[batch_idx],
          attentions[batch_idx])